# 06 — Full DDPM: Diffusion Transformer (DiT)

You now have all the ingredients — forward process, denoiser training, DDIM sampling, and classifier-free guidance. In this final notebook you replace the MLP denoiser with a **Diffusion Transformer (DiT)** and train it end-to-end with class conditioning.

New in this notebook:
- **Flax NNX** — a modern JAX neural-network library for writing cleaner model code
- **Patch embeddings** — convert 32×32 images into sequences of tokens
- **Adaptive Layer Norm (AdaLN)** — inject time and class conditioning into each Transformer block
- **CIFAR-10 class-conditional generation** trained from scratch


In [4]:
# @title Setup — run this cell first (Colab only)
!git clone https://github.com/maigimenez/let-it-rip
%cd let-it-rip
!pip install -q "jax[cuda12]" plotly jaxtyping optax flax datasets pillow


/content/let-it-rip


In [2]:
import sys
import os

# Ensure repo root is on the path so `solutions` is importable when the
# notebook is opened directly from the notebooks/ directory.
_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
for _p in [os.getcwd(), _root]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np
import optax
import flax.nnx as nnx
import pickle
from datasets import load_dataset
from jaxtyping import Array, Float, Int, PRNGKeyArray
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from PIL import Image as PILImage

print(f"JAX {jax.__version__} · backend: {jax.default_backend()}")

JAX 0.7.2 · backend: tpu


In [5]:
from solutions.schedule import cosine_schedule, q_sample
from solutions.diffusion import sinusoidal_embedding
from solutions.sampling import ddim_step

PATCH_SIZE = 4
IMAGE_SIZE = 32
N_PATCHES = (IMAGE_SIZE // PATCH_SIZE) ** 2  # 64
D_MODEL = 256
N_HEADS = 4
DEPTH = 6
N_CLASSES = 10
NULL_CLASS = 10
T = 1000

schedule = cosine_schedule(T)

In [20]:
from huggingface_hub import whoami
from google.colab import userdata

import os, requests
os.environ["HF_TOKEN"] = "userdata.get("HF_TOKEN")"
print(whoami()["name"])   # your HF username → token works; error → not set/invalid

r = requests.get("https://huggingface.co/api/whoami-v2",
                 headers={"Authorization": f"Bearer {os.environ['HF_TOKEN']}"})
print(r.status_code, {k: v for k, v in r.headers.items() if "ratelimit" in k.lower()})


CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

ds = load_dataset("uoft-cs/cifar10", split="train")
images = jnp.array(
    np.stack([np.array(ex["img"]) for ex in ds]).astype(np.float32) / 127.5 - 1.0
)
labels = jnp.array([ex["label"] for ex in ds], dtype=jnp.int32)
print(f"images: {images.shape}  labels: {labels.shape}")

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/23.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

images: (50000, 32, 32, 3)  labels: (50000,)


---
## 1 — Flax NNX in 5 minutes

Until now, model parameters were plain Python dicts. **Flax NNX** lets you define models as classes, keeping the same JAX-native approach while handling parameter bookkeeping automatically.

Key ideas:
- A model is a class inheriting from `nnx.Module`
- Sub-modules (`nnx.Linear`, `nnx.LayerNorm`, …) are assigned as attributes in `__init__`
- Every module receives `rngs=nnx.Rngs(seed)` for reproducible initialisation
- The forward pass goes in `__call__`
- `nnx.jit` and `nnx.value_and_grad` work like their JAX counterparts but handle mutable state


In [ ]:
# Given — a minimal NNX module to read before the exercise.
class LinearLayer(nnx.Module):
    def __init__(self, d_in: int, d_out: int, rngs: nnx.Rngs):
        self.layer = nnx.Linear(d_in, d_out, rngs=rngs)

    def __call__(self, x: Float[Array, " d_in"]) -> Float[Array, " d_out"]:
        return self.layer(x)


m = LinearLayer(4, 8, rngs=nnx.Rngs(0))
print(m(jnp.ones(4)).shape)  # (8,)
print(type(m.layer.kernel))  # nnx.Param — wraps a JAX array

### Exercise 1 — implement `TimestepEmbedder`

Define an NNX module that maps a **batch** of integer timesteps `t: [B]` to embeddings `[B, D_MODEL]`:

1. `__init__`: store one `nnx.Linear(d, d)` projection layer and the dimension `d`
2. `__call__`:
   - Vectorise `sinusoidal_embedding` over the batch:
     `jax.vmap(lambda ti: sinusoidal_embedding(ti, self.d))(t)`
   - Apply the projection layer (it handles the batch dim automatically)
   - Return `jnp.tanh(...)` of the result

This module is used directly inside the DiT.


In [ ]:
class TimestepEmbedder(nnx.Module):
    def __init__(self, d: int, rngs: nnx.Rngs):
        # YOUR CODE HERE
        raise NotImplementedError

    def __call__(self, t: Int[Array, " b"]) -> Float[Array, "b d"]:
        # YOUR CODE HERE
        raise NotImplementedError

In [ ]:
# @title 💡 Solution (open to reveal after trying to implement)


class TimestepEmbedder(nnx.Module):
    def __init__(self, d: int, rngs: nnx.Rngs):
        self.proj = nnx.Linear(d, d, rngs=rngs)
        self.d = d

    def __call__(self, t: Int[Array, " b"]) -> Float[Array, "b d"]:
        embs = jax.vmap(lambda ti: sinusoidal_embedding(ti, self.d))(t)
        return jnp.tanh(self.proj(embs))

In [ ]:
te = TimestepEmbedder(D_MODEL, rngs=nnx.Rngs(0))
out = te(jnp.array([0, 500, 999]))
assert out.shape == (3, D_MODEL)
assert not jnp.allclose(out[0], out[1])
print("✓ TimestepEmbedder works")

---
## 2 — Patching images

A Transformer operates on sequences. To apply it to images we split each 32×32 image into a grid of **4×4 patches** — giving 64 patches of 48 values each (4×4×3 pixels). Each patch is then linearly projected to `D_MODEL = 256`.

```
[32, 32, 3]  →  [64, 48]  →  [64, 256]    PatchEmbed
[64, 256]    →  [64, 48]  →  [32, 32, 3]  PatchUnembed
```


### Exercise 2 — implement `PatchEmbed.__call__`

The `__init__` is given. Implement `__call__` which takes `x: [B, H, W, C]` and returns `[B, N, D]`.

Steps:
1. Reshape into a patch grid: `x.reshape(B, H//p, p, W//p, p, C)`
2. Transpose to group spatial positions together: `(0, 1, 3, 2, 4, 5)` → `[B, H//p, W//p, p, p, C]`
3. Flatten each patch: `.reshape(B, -1, p*p*C)` → `[B, N, p*p*C]`
4. Apply `self.proj`


In [ ]:
class PatchEmbed(nnx.Module):
    def __init__(self, patch_size: int, in_ch: int, d: int, rngs: nnx.Rngs):
        self.patch_size = patch_size
        self.proj = nnx.Linear(patch_size * patch_size * in_ch, d, rngs=rngs)

    def __call__(self, x: Float[Array, "b h w c"]) -> Float[Array, "b n d"]:
        # YOUR CODE HERE
        raise NotImplementedError

In [ ]:
# @title 💡 Solution (open to reveal after trying to implement)


class PatchEmbed(nnx.Module):
    def __init__(self, patch_size: int, in_ch: int, d: int, rngs: nnx.Rngs):
        self.patch_size = patch_size
        self.proj = nnx.Linear(patch_size * patch_size * in_ch, d, rngs=rngs)

    def __call__(self, x: Float[Array, "b h w c"]) -> Float[Array, "b n d"]:
        B, H, W, C = x.shape
        p = self.patch_size
        x = x.reshape(B, H // p, p, W // p, p, C)
        x = x.transpose(0, 1, 3, 2, 4, 5)  # [B, H//p, W//p, p, p, C]
        x = x.reshape(B, -1, p * p * C)  # [B, N, p*p*C]
        return self.proj(x)

In [ ]:
# Given — the inverse of PatchEmbed.
class PatchUnembed(nnx.Module):
    def __init__(
        self, patch_size: int, out_ch: int, d: int, n_per_side: int, rngs: nnx.Rngs
    ):
        self.patch_size = patch_size
        self.n_per_side = n_per_side
        self.proj = nnx.Linear(d, patch_size * patch_size * out_ch, rngs=rngs)

    def __call__(self, x: Float[Array, "b n d"]) -> Float[Array, "b h w c"]:
        B, p, G = x.shape[0], self.patch_size, self.n_per_side
        x = self.proj(x)  # [B, G*G, p*p*C]
        x = x.reshape(B, G, G, p, p, -1)  # [B, G, G, p, p, C]
        x = x.transpose(0, 1, 3, 2, 4, 5)  # [B, G, p, G, p, C]
        return x.reshape(B, G * p, G * p, -1)  # [B, H, W, C]

In [ ]:
_rngs = nnx.Rngs(0)
_embed = PatchEmbed(PATCH_SIZE, 3, D_MODEL, rngs=_rngs)
_unembed = PatchUnembed(PATCH_SIZE, 3, D_MODEL, IMAGE_SIZE // PATCH_SIZE, rngs=_rngs)
_batch = images[:4]
_tokens = _embed(_batch)
_recon = _unembed(_tokens)
assert _tokens.shape == (4, N_PATCHES, D_MODEL)
assert _recon.shape == _batch.shape
print(f"PatchEmbed:   {_batch.shape} → {_tokens.shape}")
print(f"PatchUnembed: {_tokens.shape} → {_recon.shape}")
print("✓ Patching roundtrip works")

---
## 3 — DiT Block

Each DiT block is a standard Transformer block with one change: **Adaptive Layer Norm (AdaLN)**. Instead of fixed scale/shift parameters in the layer norm, AdaLN computes them from the conditioning signal (time + class embedding).

```
cond [B,d] ──► Linear(d→4d) ──► shift₁, scale₁, shift₂, scale₂   (each [B,1,d])
                                          ↓
x [B,N,d] ──► LayerNorm ──► ×(1+scale₁)+shift₁ ──► SelfAttention ──► + x
          └──► LayerNorm ──► ×(1+scale₂)+shift₂ ──► MLP ────────────► + x
```


In [ ]:
# Given — standard multi-head self-attention.
class SelfAttention(nnx.Module):
    def __init__(self, d: int, n_heads: int, rngs: nnx.Rngs):
        self.n_heads = n_heads
        self.head_dim = d // n_heads
        self.qkv = nnx.Linear(d, 3 * d, rngs=rngs)
        self.out = nnx.Linear(d, d, rngs=rngs)

    def __call__(self, x: Float[Array, "b n d"]) -> Float[Array, "b n d"]:
        B, N, d = x.shape
        H, Dh = self.n_heads, self.head_dim
        q, k, v = jnp.split(self.qkv(x), 3, axis=-1)
        q = q.reshape(B, N, H, Dh).transpose(0, 2, 1, 3)
        k = k.reshape(B, N, H, Dh).transpose(0, 2, 1, 3)
        v = v.reshape(B, N, H, Dh).transpose(0, 2, 1, 3)
        attn = jax.nn.softmax(q @ k.transpose(0, 1, 3, 2) / jnp.sqrt(Dh), axis=-1)
        out = (attn @ v).transpose(0, 2, 1, 3).reshape(B, N, d)
        return self.out(out)

### Exercise 3 — implement `AdaLNBlock.__call__`

The `__init__` is given. Implement `__call__`:

- `x: [B, N, d]` — token sequence
- `cond: [B, d]` — time + class conditioning

Steps:
1. `mod = self.adaLN(cond)` → `[B, 4d]`; split into `shift₁, scale₁, shift₂, scale₂` (each `[B, d]`)
2. Unsqueeze each to `[B, 1, d]` so they broadcast over the N dimension
3. Attention sub-layer: `x = x + self.attn(self.norm1(x) * (1 + scale₁) + shift₁)`
4. MLP sub-layer: `x = x + self.mlp2(gelu(self.mlp1(self.norm2(x) * (1 + scale₂) + shift₂)))`


In [ ]:
class AdaLNBlock(nnx.Module):
    def __init__(self, d: int, n_heads: int, rngs: nnx.Rngs):
        self.norm1 = nnx.LayerNorm(d, use_bias=False, use_scale=False, rngs=rngs)
        self.attn = SelfAttention(d, n_heads, rngs=rngs)
        self.norm2 = nnx.LayerNorm(d, use_bias=False, use_scale=False, rngs=rngs)
        self.mlp1 = nnx.Linear(d, 4 * d, rngs=rngs)
        self.mlp2 = nnx.Linear(4 * d, d, rngs=rngs)
        self.adaLN = nnx.Linear(d, 4 * d, rngs=rngs)

    def __call__(
        self,
        x: Float[Array, "b n d"],
        cond: Float[Array, "b d"],
    ) -> Float[Array, "b n d"]:
        # YOUR CODE HERE
        raise NotImplementedError

In [ ]:
# @title 💡 Solution (open to reveal after trying to implement)


class AdaLNBlock(nnx.Module):
    def __init__(self, d: int, n_heads: int, rngs: nnx.Rngs):
        self.norm1 = nnx.LayerNorm(d, use_bias=False, use_scale=False, rngs=rngs)
        self.attn = SelfAttention(d, n_heads, rngs=rngs)
        self.norm2 = nnx.LayerNorm(d, use_bias=False, use_scale=False, rngs=rngs)
        self.mlp1 = nnx.Linear(d, 4 * d, rngs=rngs)
        self.mlp2 = nnx.Linear(4 * d, d, rngs=rngs)
        self.adaLN = nnx.Linear(d, 4 * d, rngs=rngs)

    def __call__(
        self,
        x: Float[Array, "b n d"],
        cond: Float[Array, "b d"],
    ) -> Float[Array, "b n d"]:
        shift1, scale1, shift2, scale2 = jnp.split(self.adaLN(cond), 4, axis=-1)
        shift1, scale1 = shift1[:, None], scale1[:, None]  # [B, 1, d]
        shift2, scale2 = shift2[:, None], scale2[:, None]
        x = x + self.attn(self.norm1(x) * (1 + scale1) + shift1)
        x = x + self.mlp2(jax.nn.gelu(self.mlp1(self.norm2(x) * (1 + scale2) + shift2)))
        return x

---
## 4 — Full DiT

The full DiT wires all components together. The `__init__` is given — implement `__call__`.

```
xt [B,H,W,C] ──► PatchEmbed ──► [B,N,d] + pos_embed
                                      │
t  [B]  ──► TimestepEmbedder ──► [B,d] ─┐
                                          ├──► cond [B,d]
c  [B]  ──► class_embed[c]   ──► [B,d] ─┘
                                      │
                          DEPTH × AdaLNBlock(x, cond)
                                      │
                               LayerNorm ──► PatchUnembed ──► [B,H,W,C]
```


### Exercise 4 — implement `DiT.__call__`

Steps:
1. Patch-embed `xt` and add `self.pos_embed.value` (use `[None]` to broadcast over the batch)
2. Compute time embeddings: `self.time_embed(t)` → `[B, d]`
3. Look up class embeddings: `self.class_embed.value[c]` → `[B, d]`; add to time emb → `cond`
4. Run `x` through each block in `self.blocks`, passing `cond`
5. Apply `self.final_norm`, then `self.patch_unembed`


In [ ]:
class DiT(nnx.Module):
    def __init__(self, rngs: nnx.Rngs):
        self.patch_embed = PatchEmbed(PATCH_SIZE, 3, D_MODEL, rngs=rngs)
        self.patch_unembed = PatchUnembed(
            PATCH_SIZE, 3, D_MODEL, IMAGE_SIZE // PATCH_SIZE, rngs=rngs
        )
        self.pos_embed = nnx.Param(jnp.zeros((N_PATCHES, D_MODEL)))
        self.class_embed = nnx.Param(
            jr.normal(rngs.params(), (NULL_CLASS + 1, D_MODEL)) * 0.02
        )
        self.time_embed = TimestepEmbedder(D_MODEL, rngs=rngs)
        self.blocks = nnx.List(
            [AdaLNBlock(D_MODEL, N_HEADS, rngs=rngs) for _ in range(DEPTH)]
        )
        self.final_norm = nnx.LayerNorm(D_MODEL, rngs=rngs)

    def __call__(
        self,
        xt: Float[Array, "b h w c"],
        t: Int[Array, " b"],
        c: Int[Array, " b"],
    ) -> Float[Array, "b h w c"]:
        # YOUR CODE HERE
        raise NotImplementedError

In [ ]:
# @title 💡 Solution (open to reveal after trying to implement)


class DiT(nnx.Module):
    def __init__(self, rngs: nnx.Rngs):
        self.patch_embed = PatchEmbed(PATCH_SIZE, 3, D_MODEL, rngs=rngs)
        self.patch_unembed = PatchUnembed(
            PATCH_SIZE, 3, D_MODEL, IMAGE_SIZE // PATCH_SIZE, rngs=rngs
        )
        self.pos_embed = nnx.Param(jnp.zeros((N_PATCHES, D_MODEL)))
        self.class_embed = nnx.Param(
            jr.normal(rngs.params(), (NULL_CLASS + 1, D_MODEL)) * 0.02
        )
        self.time_embed = TimestepEmbedder(D_MODEL, rngs=rngs)
        self.blocks = nnx.List(
            [AdaLNBlock(D_MODEL, N_HEADS, rngs=rngs) for _ in range(DEPTH)]
        )
        self.final_norm = nnx.LayerNorm(D_MODEL, rngs=rngs)

    def __call__(
        self,
        xt: Float[Array, "b h w c"],
        t: Int[Array, " b"],
        c: Int[Array, " b"],
    ) -> Float[Array, "b h w c"]:
        x = self.patch_embed(xt) + self.pos_embed.value[None]
        cond = self.time_embed(t) + self.class_embed.value[c]
        for block in self.blocks:
            x = block(x, cond)
        return self.patch_unembed(self.final_norm(x))

In [ ]:
model = DiT(rngs=nnx.Rngs(0))
x_test = images[:4]
t_test = jnp.array([100, 200, 300, 400])
c_test = jnp.array([0, 1, 2, 3])
out = model(x_test, t_test, c_test)
assert out.shape == x_test.shape
print(f"✓ DiT forward: {x_test.shape} → {out.shape}")

---
## 5 — Training

NNX training differs from the pure-JAX approach in two ways:
- `nnx.value_and_grad` instead of `jax.value_and_grad` — it understands mutable module state
- `nnx.Optimizer` wraps both the model and the optimizer state;   `optimizer.update(grads)` updates the model parameters in place

The CFG label dropout from notebook 05 is unchanged.


In [ ]:
optimizer = nnx.Optimizer(model, optax.adam(1e-4), wrt=nnx.Param)

### Exercise 5 — implement `train_step`

Steps:
1. Split `key` into `key_t`, `key_noise`, `key_mask`
2. Sample random timesteps `ts: [B]` and noise `[B, H, W, C]`
3. Sample a CFG dropout mask with `jr.bernoulli`; replace masked labels with `NULL_CLASS`
4. Compute `xt_batch` by vmapping `q_sample` over the batch
5. Define `loss_fn(model)`: call `model(xt_batch, ts, labels_masked)`, return MSE vs `noise`
6. `loss, grads = nnx.value_and_grad(loss_fn)(model)` then `optimizer.update(model, grads)`


In [ ]:
@nnx.jit
def train_step(
    model: DiT,
    optimizer: nnx.Optimizer,
    x0_batch: Float[Array, "b h w c"],
    labels_batch: Int[Array, " b"],
    key: PRNGKeyArray,
    schedule: dict,
    p_uncond: float = 0.1,
) -> Float[Array, ""]:
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# @title 💡 Solution (open to reveal after trying to implement)


@nnx.jit
def train_step(
    model: DiT,
    optimizer: nnx.Optimizer,
    x0_batch: Float[Array, "b h w c"],
    labels_batch: Int[Array, " b"],
    key: PRNGKeyArray,
    schedule: dict,
    p_uncond: float = 0.1,
) -> Float[Array, ""]:
    B = x0_batch.shape[0]
    key_t, key_noise, key_mask = jr.split(key, 3)
    ts = jr.randint(key_t, (B,), 0, T)
    noise = jr.normal(key_noise, x0_batch.shape)
    mask = jr.bernoulli(key_mask, p_uncond, (B,))
    labels_masked = jnp.where(mask, NULL_CLASS, labels_batch)

    xt_batch = jax.vmap(q_sample, in_axes=(0, 0, 0, None))(
        x0_batch, ts, noise, schedule["alphas_bar"]
    )

    def loss_fn(model):
        eps_pred = model(xt_batch, ts, labels_masked)
        return jnp.mean((noise - eps_pred) ** 2)

    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(model, grads)
    return loss

In [ ]:
def train(model, optimizer, images, labels, key, n_epochs=20, batch_size=128):
    n = len(images)
    history = []
    for epoch in range(1, n_epochs + 1):
        key, shuffle_key = jr.split(key)
        perm = jr.permutation(shuffle_key, n)
        imgs_s = images[perm]
        lbls_s = labels[perm]
        epoch_losses = []
        for i in range(0, n - batch_size + 1, batch_size):
            key, step_key = jr.split(key)
            loss = train_step(
                model,
                optimizer,
                imgs_s[i : i + batch_size],
                lbls_s[i : i + batch_size],
                step_key,
                schedule,
            )
            epoch_losses.append(float(loss))
        mean_loss = float(np.mean(epoch_losses))
        history.append(mean_loss)
        print(f"epoch {epoch:3d} | loss {mean_loss:.4f}")
    return history


key = jr.PRNGKey(0)
history = train(model, optimizer, images, labels, key, n_epochs=20)

In [ ]:
fig = go.Figure(go.Scatter(y=history, mode="lines+markers"))
fig.update_layout(
    title="Training loss", xaxis_title="epoch", yaxis_title="MSE", height=350
)
fig.show()

In [ ]:
state = nnx.state(model)
with open("params_nb06.pkl", "wb") as f:
    pickle.dump(state, f)
print("Saved params_nb06.pkl ✓")

---
## 6 — Sampling

DDIM + CFG sampling is identical to notebook 05 — only the model call signature changed (the DiT takes batched `t` and `c` directly).


In [ ]:
def upscale(img_uint8, scale=4):
    h, w = img_uint8.shape[:2]
    return np.array(
        PILImage.fromarray(img_uint8).resize((w * scale, h * scale), PILImage.LANCZOS)
    )


def to_uint8(arr):
    return np.clip((np.array(arr) + 1) / 2 * 255, 0, 255).astype(np.uint8)


def cfg_noise_estimate(model, xt, t, c, w=3.0):
    t_b = jnp.array([t])
    eps_u = model(xt[None], t_b, jnp.array([NULL_CLASS]))[0]
    eps_c = model(xt[None], t_b, jnp.array([c]))[0]
    return eps_u + w * (eps_c - eps_u)


def ddim_sample_cfg(model, c, key, w=3.0, steps=50, shape=(32, 32, 3)):
    timesteps = np.linspace(T - 1, 0, steps + 1).round().astype(int)
    xt = jr.normal(key, shape)
    for i in range(len(timesteps) - 1):
        t_from, t_to = int(timesteps[i]), int(timesteps[i + 1])
        xt = ddim_step(
            xt, t_from, t_to, cfg_noise_estimate(model, xt, t_from, c, w), schedule
        )
    return xt

In [ ]:
N_COLS, W = 4, 3.0
fig = make_subplots(
    rows=10,
    cols=N_COLS,
    row_titles=CLASS_NAMES,
    horizontal_spacing=0.02,
    vertical_spacing=0.03,
)
key = jr.PRNGKey(42)
for row, class_id in enumerate(range(10), start=1):
    for col in range(1, N_COLS + 1):
        key, sk = jr.split(key)
        img = ddim_sample_cfg(model, class_id, sk, w=W, steps=50)
        fig.add_trace(go.Image(z=upscale(to_uint8(img))), row=row, col=col)
fig.update_xaxes(showticklabels=False).update_yaxes(showticklabels=False)
fig.update_layout(
    title=f"DiT CIFAR-10 — class-conditional generation (w={W})", height=1400
)
fig.show()

In [ ]:
class_id = 0  # airplane
w_values = [1.0, 2.0, 4.0, 7.0, 12.0]

fig = make_subplots(
    rows=1,
    cols=len(w_values),
    subplot_titles=[f"w = {w}" for w in w_values],
    horizontal_spacing=0.02,
)
key = jr.PRNGKey(0)
for col, w in enumerate(w_values, start=1):
    key, sk = jr.split(key)
    img = ddim_sample_cfg(model, class_id, sk, w=w, steps=50)
    fig.add_trace(go.Image(z=upscale(to_uint8(img), scale=4)), row=1, col=col)
fig.update_xaxes(showticklabels=False).update_yaxes(showticklabels=False)
fig.update_layout(title=f"Guidance scale sweep — {CLASS_NAMES[class_id]}", height=300)
fig.show()

---
## What you built

| Notebook | What you built |
|----------|----------------|
| 01 | JAX fundamentals: jit, vmap, PRNG |
| 02 | Forward process: noise schedules, closed-form sampling |
| 03 | Reverse process: MLP denoiser, training loop |
| 04 | Sampling: DDPM and DDIM |
| 05 | Class-conditional generation with CFG |
| **06** | **DiT: patch embeddings, AdaLN, NNX training** |

The architecture you just trained — DiT + cosine schedule + DDIM + CFG — is the core of modern text-to-image models like Stable Diffusion 3 and FLUX, scaled up and trained on billions of images.
